# AI Startup Funding Rounds 2021–2026: starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/ai_startup_funding_sample.csv', recursive=True) or ['ai_startup_funding_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
by_year = df.groupby('year').agg(rounds=('round_id', 'size'), usd_bn=('amount_usd', lambda s: s.sum() / 1e9))
by_year.usd_bn.plot.bar(figsize=(9, 3.5), color='#2563eb', title='Disclosed funding by year (sample, US$ bn)')
by_year

In [ ]:
df[df.stage.isin(['Seed', 'Series A', 'Series B'])].groupby(['year', 'stage']).amount_usd.median().unstack().div(1e6).round(2)

In [ ]:
df.lead_investor.replace('', None).value_counts().head(20)

## Get the full record from the Radar API

The sample keeps one teaser field (`lead_investor`). The API returns every investor, valuations and the full round history. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['company_domain'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/companies/funding', headers={'Authorization': f'Bearer {key}'},
                      json={'domain': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp